# §26 — Cubic'i Kendi Evinde Test: ekstrapolasyon (seyrek + uzun-ömür)

Cubic'in yapısal avantajı (§15h'de graft-yoğun rejimde kayboldu) burada aranır:
seyrek yazım + eğitilen ufkun ÖTESİNDE geri getirme. Vasıta = borusu döşenmiş
cross-chunk carry hattı. Twin: exp vs cubic_flux_chunked, 3 seed. Ayırt edici
sinyal: K=32,64 (eğitilen ufuk 16'nın 2-4 katı = ekstrapolasyon).

**Qwen YOK, token YOK, indirme YOK** — küçük-ölçek, sıfırdan; CPU'da bile koşar.
Ön-kayıtlı kriterler RESULTS §26'da. ~20-40 dk (T4) / biraz daha uzun (CPU).


In [ ]:
# --- 1. KURULUM (Qwen'siz, hafif) ---
import os, subprocess, sys
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
try:
    import torch; print('torch var:', torch.__version__)
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','torch','numpy'],check=True)
CKDIR = os.path.join(BASE,'cubic_niche_ckpt'); os.makedirs(CKDIR, exist_ok=True)
print('repo:', REPO, '| ckpt:', CKDIR)

In [ ]:
# --- 2. EGITIM: carry_curriculum (exp & cubic x 3 seed), CARRY_MAX=16 ---
import os, subprocess, time
env = {**os.environ, 'PYTHONPATH': REPO, 'HFP_CKPT_DIR': CKDIR,
       'CC_CARRY_MAX':'16', 'CC_STEPS':'1200', 'CC_CTX':'256', 'CC_P':'6',
       'CC_DIST_EVERY':'64', 'CC_BS':'8', 'CC_GAPS':'256,4096,16384'}
MODES = ['exp','cubic_flux_chunked']; SEEDS = [0,1,2]
for mode in MODES:
    for seed in SEEDS:
        ck = f'{CKDIR}/carryv1_{mode}_s{seed}.pt'
        if os.path.exists(ck):
            print(f'[atla] {mode} s{seed} zaten egitilmis'); continue
        print(f'\n===== EGITIM {mode} s{seed} =====', flush=True)
        t=time.time()
        # butce genis: tek celiste bitsin (bitmezse resume icin tekrar kosman yeter)
        r = subprocess.run([sys.executable,'review_scripts/carry_curriculum.py',mode,str(seed),'6000'],
                           cwd=REPO, env=env)
        print(f'  ({(time.time()-t)/60:.1f} dk, exit {r.returncode})', flush=True)
print('\nEGITIM TAMAM')

In [ ]:
# --- 3. EVAL: matched_probe (temiz probe), K=0..64 (32,64 = ekstrapolasyon) ---
import os, subprocess
env = {**os.environ, 'PYTHONPATH': REPO, 'HFP_CKPT_DIR': CKDIR,
       'MP_KS':'0,2,8,16,32,64', 'MP_TRIALS':'50', 'MP_CTX':'256', 'MP_P':'6', 'MP_DIST_EVERY':'64'}
for mode in ['exp','cubic_flux_chunked']:
    for seed in [0,1,2]:
        print(f'\n===== EVAL {mode} s{seed} =====', flush=True)
        subprocess.run([sys.executable,'review_scripts/matched_probe.py',mode,str(seed)], cwd=REPO, env=env)
print('\nEVAL TAMAM')

In [ ]:
# --- 4. TOPLA + ON-KAYITLI HUKUM (§26) ---
import csv, os, numpy as np
KS=[0,2,8,16,32,64]
def load(mode):
    accs={k:[] for k in KS}
    for seed in [0,1,2]:
        f=f'{CKDIR}/matchedv1_{mode}_s{seed}.csv'
        if not os.path.exists(f): print('EKSIK:',f); continue
        for row in csv.DictReader(open(f)):
            k=int(row['K'])
            if k in accs: accs[k].append(float(row['matched_acc']))
    return accs
E=load('exp'); C=load('cubic_flux_chunked')
print(f"{'K':>4} {'~token':>8} | {'exp acc':>18} | {'cubic acc':>18} | {'fark':>7}")
print('-'*64)
for k in KS:
    e=E[k]; c=C[k]
    em,cm=np.mean(e) if e else float('nan'), np.mean(c) if c else float('nan')
    tag=' <-EKSTRAPOLASYON' if k>16 else (' (egitim ufku)' if k==16 else '')
    print(f'{k:>4} {k*256:>8} | {em:6.1f}% [{min(e):.0f}-{max(e):.0f}] | '
          f'{cm:6.1f}% [{min(c):.0f}-{max(c):.0f}] | {cm-em:+6.1f}{tag}')

print('\n=== ON-KAYITLI HUKUM (§26) — sans %3.3 ===')
def sep(k):  # seed araliklari ayrisiyor mu (cubic > exp)
    return min(C[k])>max(E[k])
def verdict():
    ext=[32,64]
    c_adv=all((np.mean(C[k])-np.mean(E[k]))>=10 and sep(k) for k in ext)
    e_adv=all((np.mean(E[k])-np.mean(C[k]))>=10 and min(E[k])>max(C[k]) for k in ext)
    if c_adv:  return 'CUBIC NISI DOGRULANDI: K=32 VE 64\'te cubic >= exp+10 puan, araliklar ayrik. Cubic kendi evinde kazaniyor -> seyrek/uzun-omur (cihaz-ici hafiza) rejimi icin cubic gecerli.'
    if e_adv:  return 'EXP kendi evinde bile onde -> cubic emekli, exp her yerde varsayilan.'
    d32=np.mean(C[32])-np.mean(E[32]); d64=np.mean(C[64])-np.mean(E[64])
    return (f'NULL: net nis yok (K32 fark {d32:+.1f}, K64 fark {d64:+.1f}). Cubic tahmin '
            f'edilen ev rejiminde bile belirgin ustunluk gostermedi -> exp durust varsayilan.')
print(verdict())